# Explore a frozen search comparison
This notebook reads the retained public API comparison report. Its output is descriptive evidence; it cannot change the comparison verdict.

In [ ]:
comparison_path = 'comparison.json'


In [ ]:
import json
from pathlib import Path
from statistics import mean

report = json.loads(Path(comparison_path).read_text(encoding='utf-8'))
rows = report.get('queries', [])
print('Mode:', report.get('mode', report.get('kind', 'unknown')))
print('Verdict:', report.get('verdict', 'see report'))
print('Queries in report:', report.get('query_count', len(rows)))
if report.get('kind') == 'offline-evaluation-report':
    for side in ('baseline', 'candidate'):
        print(side.title(), 'metrics =', report['metrics'][side],
              'judged coverage =', report['coverage'][side].get('fraction'))
    for metric, candidate in report['metrics']['candidate'].items():
        baseline = report['metrics']['baseline'].get(metric)
        if isinstance(candidate, (int, float)) and isinstance(baseline, (int, float)):
            print(metric, 'candidate minus baseline =', round(candidate - baseline, 6))
elif report.get('kind') == 'variant-evaluation-report':
    print('Default variant:', report['default_variant'])
    print('Metric baseline:', report['baseline_variant'])
    for variant in report['variants']:
        print('Variant', variant, 'metrics =', report['metrics'][variant],
              'delta =', report.get('delta_from_baseline', {}).get(variant, {}),
              'judged coverage =', report['coverage'][variant].get('fraction'),
              'changed query fraction =', report['result_changes'][variant]['fraction'])
elif rows:
    changed = [row for row in rows if row.get('equal_top_10') is False]
    similarities = [row['jaccard_at_10'] for row in rows if isinstance(row.get('jaccard_at_10'), (int, float))]
    print('Changed top-10 lists:', len(changed))
    if similarities:
        print('Mean Jaccard@10:', round(mean(similarities), 4))
    print('Examples of changed queries:')
    for row in changed[:10]:
        print(' ', row['query_id'], row.get('query', ''), 'Jaccard@10 =', row.get('jaccard_at_10'))
    scored = [row for row in rows if isinstance(row.get('ndcg_delta_at_10'), (int, float))]
    if scored:
        print('Largest measured nDCG@10 losses:')
        for row in sorted(scored, key=lambda item: item['ndcg_delta_at_10'])[:5]:
            print(' ', row['query_id'], row.get('query', ''), 'delta =', row['ndcg_delta_at_10'])
else:
    print('No per-query rows; inspect the retained report for phase-level measurements.')
